In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/cohort-x-task-1/Task_1.xlsx
/kaggle/input/competitions/cohort-x-task-1/PMC_NXML_Archives/PMC11817859.nxml
/kaggle/input/competitions/cohort-x-task-1/PMC_NXML_Archives/PMC10503118.nxml
/kaggle/input/competitions/cohort-x-task-1/PMC_NXML_Archives/PMC10410925.nxml
/kaggle/input/competitions/cohort-x-task-1/PMC_NXML_Archives/PMC12681079.nxml
/kaggle/input/competitions/cohort-x-task-1/PMC_NXML_Archives/PMC12649937.nxml
/kaggle/input/competitions/cohort-x-task-1/PMC_NXML_Archives/PMC12132709.nxml
/kaggle/input/competitions/cohort-x-task-1/PMC_NXML_Archives/PMC12622353.nxml
/kaggle/input/competitions/cohort-x-task-1/PMC_NXML_Archives/PMC12628283.nxml
/kaggle/input/competitions/cohort-x-task-1/PMC_NXML_Archives/PMC9811764.nxml
/kaggle/input/competitions/cohort-x-task-1/PMC_NXML_Archives/PMC12673831.nxml
/kaggle/input/competitions/cohort-x-task-1/PMC_NXML_Archives/PMC10230393.nxml
/kaggle/input/competitions/cohort-x-task-1/PMC_NXML_Archives/PMC12248779.nxml
/kaggle/in

In [2]:
# ============================================================================
# CohortX Task 1 - Pipeline v10  (RULE-COMPLIANT, EXTRACTION-ONLY)
# ============================================================================
# Paste this ENTIRE thing into ONE Kaggle cell and run. Nothing else needed.
#
# *** RULES COMPLIANCE ***
# The competition host stated on the forum:
#   "Are publicly available external datasets (e.g. PubMed, ClinicalTrials.gov,
#    UMLS, MeSH...) allowed for training or inference?  No. Absolutely not.
#    The only data you can use is the one we provided to you on Kaggle."
#   "You are not permitted to use any external services like APIs."
#   Lightweight pretrained models ARE allowed.
# Therefore this pipeline uses ONLY:
#   - the provided Task_1.xlsx (train labels) and the provided PMC NXML files
#   - a lightweight pretrained sentence model (explicitly permitted)
# There is NO ClinicalTrials.gov / AACT lookup, NO external dataset, NO API call.
#
# *** STRATEGY (taken from the host's own guidance) ***
#   Host: "the task ... [is] to obtain the most similar piece from the NXML file"
# So eligibility is a PASSAGE-SELECTION problem: cut the paper into candidate
# passages, then RANK them and keep the best. We LEARN that ranking from the 416
# provided training papers: for each candidate we compute its TRUE FM3S score
# against the gold, then fit a small model to predict that score from cheap
# features. All learning uses provided data only.
#
# Contents:
#   1. Real evaluation harness (FM3S + BioBERT cosine + number Jaccard) on a 22%
#      holdout, so every change is measured before it is kept
#   2. Base rule-based extraction (eligibility / age / sex / conditions / type)
#   3. NEW: candidate passage generation + LEARNED eligibility ranker
#   4. NEW: conservative age fixes (ignore result-stats, case-reports, animals)
#   5. A/B tests everything on the holdout; keeps only what wins
#   6. Writes submission.csv with all 6 required columns filled
#
# SAFETY GUARD: every new method is compared against the current one on the
# holdout and used ONLY if it scores at least as high. Nothing can silently
# lower your score.
# ============================================================================

import ast
import re
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
BASE_DIR = Path("/kaggle/input/competitions/cohort-x-task-1")
XLSX_PATH = BASE_DIR / "Task_1.xlsx"
XML_DIR = BASE_DIR / "PMC_NXML_Archives"
OUT_DIR = Path("/kaggle/working")
SUBMISSION_PATH = OUT_DIR / "submission.csv"

BERT_MODEL_NAME = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"  # >>> SWAP POINT
SCISPACY_MODEL = "en_ner_bc5cdr_md"   # optional; auto-fallback if absent

ELIG_CAP_CANDIDATES = [400, 600, 800, 1000, 1400]   # base-rule sweep (unchanged)
ELIG_CAP_WIDE = [1000, 1400, 1800, 2200, 2600, 3000]  # v12: the trend said LONGER is better
W_ELIG, W_COND, W_STUDY, W_NUM = 0.50, 0.20, 0.15, 0.15   # >>> SWAP POINT
FORMAT_AS_LIST = True

AGE_MIN_DEFAULT = "18 Years"
AGE_MAX_DEFAULT = "Not Specified"

# --- v10: learned eligibility ranker settings ---
RANKER_TRAIN_PAPERS = 180        # papers used to build the ranker training signal
RANKER_TRAIN_PAPERS_FINAL = 320  # papers used to retrain the ranker for the final model
TOPK_CANDIDATES = [4, 6, 10, 14, 18, 24]  # v12: widened (more passages = more words to match)

# --- v11: learned conditions ranker settings ---
COND_RANKER_PAPERS = 300         # papers used to train the conditions ranker
COND_RANKER_PAPERS_FINAL = 416   # all training papers for the final model
COND_TOPK_CANDIDATES = [1, 2, 3] # how many condition terms to keep (swept)

OUT_DIR.mkdir(parents=True, exist_ok=True)
for p in OUT_DIR.glob("*.csv"):
    p.unlink()


# ============================================================================
# SECTION 0 — Dependency bootstrap
# ============================================================================
def _ensure_nltk():
    import nltk
    for pkg, path in [
        ("wordnet", "corpora/wordnet"), ("omw-1.4", "corpora/omw-1.4"),
        ("wordnet_ic", "corpora/wordnet_ic"), ("punkt", "tokenizers/punkt"),
        ("averaged_perceptron_tagger", "taggers/averaged_perceptron_tagger"),
    ]:
        try:
            nltk.data.find(path)
        except LookupError:
            try:
                nltk.download(pkg, quiet=True)
            except Exception as e:
                print(f"[warn] nltk {pkg}: {e!r}")


_ensure_nltk()

try:
    import spacy
    try:
        _NLP = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
    except Exception:
        _NLP = spacy.blank("en")
        if "sentencizer" not in _NLP.pipe_names:
            _NLP.add_pipe("sentencizer")
    _SPACY_OK = True
except Exception as e:
    print(f"[warn] spaCy unavailable -> {e!r}; regex sentence split.")
    _NLP, _SPACY_OK = None, False

_SCISPACY = None
try:
    import spacy as _sp
    _SCISPACY = _sp.load(SCISPACY_MODEL)
    print(f"[ok] scispaCy loaded: {SCISPACY_MODEL}")
except Exception as e:
    print(f"[warn] scispaCy '{SCISPACY_MODEL}' not available -> conditions use "
          f"title/abstract matching (still works offline). ({type(e).__name__})")


# ============================================================================
# SECTION 1 — Basic helpers
# ============================================================================
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", x).strip()


def clean_pmcid(x):
    x = str(x).strip()
    x = re.sub(r"\.0$", "", x)
    return x.replace("PMC", "")


def parse_list_label(x):
    if pd.isna(x):
        return []
    text = str(x).strip()
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [clean_text(v) for v in parsed if clean_text(v)]
    except Exception:
        pass
    return [clean_text(v).strip("[]'\"") for v in re.split(r"[;,|]\s*|\n", text) if clean_text(v)]


def tokenise(text):
    return re.findall(r"[a-z0-9]+", clean_text(text).lower())


def sent_split(text):
    text = clean_text(text)
    if not text:
        return []
    if _SPACY_OK and _NLP is not None:
        try:
            return [s.text.strip() for s in _NLP(text[:100000]).sents if s.text.strip()]
        except Exception:
            pass
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]


# ============================================================================
# SECTION 2 — EVALUATION HARNESS  (reconstructed; >>> SWAP POINT throughout)
# ============================================================================

# 2a. Number similarity
_WORD2NUM_OK = False
try:
    from word2number import w2n
    _WORD2NUM_OK = True
except Exception:
    print("[warn] word2number missing (spelled-out numbers skipped; minor).")

_NUM_WORDS = (r"zero|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|"
              r"thirteen|fourteen|fifteen|sixteen|seventeen|eighteen|nineteen|twenty|"
              r"thirty|forty|fifty|sixty|seventy|eighty|ninety|hundred|thousand")


def extract_numbers(text):
    text = clean_text(text).lower()
    if not text or text == "not specified":
        return set()
    nums = set()
    for m in re.findall(r"-?\d+(?:\.\d+)?", text):
        try:
            v = float(m)
            nums.add(int(v) if v.is_integer() else round(v, 3))
        except Exception:
            pass
    if _WORD2NUM_OK:
        for m in re.findall(rf"(?:{_NUM_WORDS})(?:[\s-]+(?:{_NUM_WORDS}))*", text):
            try:
                nums.add(int(w2n.word_to_num(m)))
            except Exception:
                pass
    return nums


def number_similarity(pred, ref):
    P, R = extract_numbers(pred), extract_numbers(ref)
    if not P and not R:
        return 1.0
    if not P or not R:
        return 0.0
    return len(P & R) / len(P | R)


# 2b. BioBERT cosine
class BertScorer:
    def __init__(self, model_name):
        self.ok, self.model = False, None
        try:
            from sentence_transformers import SentenceTransformer, util
            self._util = util
            self.model = SentenceTransformer(model_name, device="cpu")
            self.ok = True
            print(f"[ok] BioBERT loaded: {model_name}")
        except Exception as e:
            print(f"[warn] BioBERT unavailable -> {e!r}; token-Jaccard fallback.")

    def sim(self, a, b):
        a, b = clean_text(a), clean_text(b)
        if not a and not b:
            return 1.0
        if not a or not b:
            return 0.0
        if self.ok:
            try:
                emb = self.model.encode([a, b], convert_to_tensor=True,
                                        normalize_embeddings=True, show_progress_bar=False)
                return float(self._util.cos_sim(emb[0], emb[1]).item())
            except Exception:
                pass
        pa, pb = set(tokenise(a)), set(tokenise(b))
        return len(pa & pb) / len(pa | pb) if (pa | pb) else 1.0


_BERT = BertScorer(BERT_MODEL_NAME)


def semantic_bert_sim(pred, ref):
    return _BERT.sim(pred, ref)


# 2c. FM3S (reconstructed)
_STOP_VERBS = {"be", "is", "are", "was", "were", "been", "being", "am", "have",
               "has", "had", "do", "does", "did", "say", "said", "get", "got",
               "make", "made", "go", "went"}

try:
    from nltk.corpus import wordnet as wn
    from nltk.corpus import wordnet_ic
    _WN_OK = True
    try:
        _BROWN_IC = wordnet_ic.ic("ic-brown.dat")
    except Exception:
        _BROWN_IC = None
except Exception:
    _WN_OK, _BROWN_IC = False, None


def _pos_tokens(text):
    text = clean_text(text)
    if not text:
        return [], []
    nouns, verbs = [], []
    if _SPACY_OK and _NLP is not None:
        try:
            for t in _NLP(text[:100000]):
                if t.pos_ in ("NOUN", "PROPN") and t.is_alpha:
                    nouns.append((t.lemma_ or t.text).lower())
                elif t.pos_ == "VERB" and t.is_alpha:
                    lemma = (t.lemma_ or t.text).lower()
                    if lemma not in _STOP_VERBS:
                        verbs.append((lemma, t.tag_))
            return nouns, verbs
        except Exception:
            pass
    try:
        import nltk
        for w, tag in nltk.pos_tag(nltk.word_tokenize(text)):
            wl = w.lower()
            if not wl.isalpha():
                continue
            if tag.startswith("NN"):
                nouns.append(wl)
            elif tag.startswith("VB") and wl not in _STOP_VERBS:
                verbs.append((wl, tag))
    except Exception:
        nouns = tokenise(text)
    return nouns, verbs


def _best_lin(a, b, pos):
    if not _WN_OK:
        return 1.0 if a == b else 0.0
    sa, sb = wn.synsets(a, pos=pos), wn.synsets(b, pos=pos)
    if not sa or not sb:
        return 1.0 if a == b else 0.0
    best = 0.0
    for x in sa[:3]:
        for y in sb[:3]:
            try:
                s = x.lin_similarity(y, _BROWN_IC) if _BROWN_IC else (x.path_similarity(y) or 0.0)
            except Exception:
                s = 0.0
            if s and s > best:
                best = s
    return best


def _avg_best(A, B, pos):
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0

    def one_way(P, Q):
        return sum(max((_best_lin(p, q, pos) for q in Q), default=0.0) for p in P) / len(P)

    return 0.5 * (one_way(A, B) + one_way(B, A))


def _noun_sim(a, b):
    na, _ = _pos_tokens(a)
    nb, _ = _pos_tokens(b)
    return _avg_best(na, nb, wn.NOUN if _WN_OK else "n")


def _verb_sim(a, b):
    _, va_ = _pos_tokens(a)
    _, vb_ = _pos_tokens(b)
    if not va_ and not vb_:
        return 1.0
    if not va_ or not vb_:
        return 0.0

    def coarse(t):
        return t[:2] if t else ""

    tags = {coarse(t) for _, t in va_} | {coarse(t) for _, t in vb_}
    scores = []
    for tg in tags:
        A = [w for w, t in va_ if coarse(t) == tg]
        B = [w for w, t in vb_ if coarse(t) == tg]
        if A and B:
            scores.append(_avg_best(A, B, wn.VERB if _WN_OK else "v"))
        elif A or B:
            scores.append(0.0)
    return float(np.mean(scores)) if scores else 0.0


def _cwo(a, b):
    ua, ub = tokenise(a), tokenise(b)
    if not ua and not ub:
        return 1.0
    if not ua or not ub:
        return 0.0
    common = set(ua) & set(ub)
    if not common:
        return 0.0
    pa = {w: i / max(len(ua) - 1, 1) for i, w in enumerate(ua) if w in common}
    pb = {w: i / max(len(ub) - 1, 1) for i, w in enumerate(ub) if w in common}
    uni = float(np.mean([1.0 - abs(pa[w] - pb[w]) for w in common]))
    biga = list(zip(ua, ua[1:]))
    bigb = set(zip(ub, ub[1:]))
    big = (sum(1 for g in biga if g in bigb) / len(biga)) if biga else 0.0
    return float(0.5 * uni + 0.5 * big)


def fm3s_similarity(pred, ref):
    pred, ref = clean_text(pred), clean_text(ref)
    if not pred and not ref:
        return 1.0
    if not pred or not ref:
        return 0.0
    if not _WN_OK:
        pa, pb = set(tokenise(pred)), set(tokenise(ref))
        return len(pa & pb) / len(pa | pb) if (pa | pb) else 1.0
    n, v, c = _noun_sim(pred, ref), _verb_sim(pred, ref), _cwo(pred, ref)
    return float(np.clip(1.0 - (1.0 - n) * (1.0 - v) ** 0.5 * (1.0 - c) ** 0.5, 0.0, 1.0))


def score_row(pred, gold):
    s_elig = fm3s_similarity(pred["eligibility_criteria"], gold["eligibility_criteria"])
    s_cond = semantic_bert_sim(pred["conditions"], gold["conditions"])
    s_study = semantic_bert_sim(pred["study_type"], gold["study_type"])
    s_num = number_similarity(f"{pred['minimum_age']} {pred['maximum_age']}",
                              f"{gold['minimum_age']} {gold['maximum_age']}")
    total = W_ELIG * s_elig + W_COND * s_cond + W_STUDY * s_study + W_NUM * s_num
    return total, {"elig": s_elig, "cond": s_cond, "study": s_study, "num": s_num}


def score_frame(pred_df, gold_df):
    comps, br = [], {"elig": [], "cond": [], "study": [], "num": []}
    for i in range(len(pred_df)):
        t, d = score_row(pred_df.iloc[i], gold_df.iloc[i])
        comps.append(t)
        for k in br:
            br[k].append(d[k])
    return float(np.mean(comps)), {k: float(np.mean(v)) for k, v in br.items()}


# ============================================================================
# SECTION 3 — Load + parse
# ============================================================================
from bs4 import BeautifulSoup  # noqa: E402

train_df = pd.read_excel(XLSX_PATH, sheet_name="Train")
test_df = pd.read_excel(XLSX_PATH, sheet_name="Test")
print("Train:", train_df.shape, "| Test:", test_df.shape)

xml_paths = {}
for p in XML_DIR.glob("*.nxml"):
    m = re.search(r"PMC(\d+)", p.name)
    if m:
        xml_paths[m.group(1)] = p
print("XML mapped:", len(xml_paths))


def parse_article(pmcid):
    pmcid = clean_pmcid(pmcid)
    path = xml_paths.get(pmcid)
    if path is None:
        return {"title": "", "abstract": "", "keywords": "", "sections": [],
                "full_text": "", "list_items": []}
    raw = path.read_text(errors="ignore")
    soup = BeautifulSoup(raw, "xml")
    title = clean_text(soup.find("article-title").get_text(" ")) if soup.find("article-title") else ""
    abstract = clean_text(soup.find("abstract").get_text(" ")) if soup.find("abstract") else ""
    keywords = "; ".join(clean_text(k.get_text(" ")) for k in soup.find_all("kwd")
                         if clean_text(k.get_text(" ")))
    sections = []
    for sec_idx, sec in enumerate(soup.find_all("sec")):
        tt = sec.find("title", recursive=False)
        sec_title = clean_text(tt.get_text(" ")) if tt else ""
        paras = []
        for pp in sec.find_all("p", recursive=False):
            txt = clean_text(pp.get_text(" "))
            if len(txt) > 35:
                paras.append(txt)
        if sec_title or paras:
            sections.append({"idx": sec_idx, "title": sec_title, "paragraphs": paras})
    # LAYOUT-AWARE: capture explicit <list-item> text (bulleted criteria, when present)
    list_items = []
    for li in soup.find_all("list-item"):
        t = clean_text(li.get_text(" "))
        if 8 <= len(t) <= 400:
            list_items.append(t)
    full_text = clean_text(" ".join([title, abstract, keywords] +
                                    [" ".join(s["paragraphs"]) for s in sections]))
    return {"title": title, "abstract": abstract, "keywords": keywords,
            "sections": sections, "full_text": full_text, "list_items": list_items}


train_df["pmcid_clean"] = train_df["pmcids"].apply(clean_pmcid)
test_df["pmcid_clean"] = test_df["pmcids"].apply(clean_pmcid)
train_docs = {pid: parse_article(pid) for pid in train_df["pmcid_clean"]}
test_docs = {pid: parse_article(pid) for pid in test_df["pmcid_clean"]}
print("Parsed:", len(train_docs), "train /", len(test_docs), "test")


# ============================================================================
# SECTION 4 — ELIGIBILITY (purify + strip dates/codes/counts)
# ============================================================================
_INCL_CUES = ["inclusion criteria", "eligible", "eligibility", "were included",
              "were enrolled", "were recruited", "must have", "were required to",
              "patients with", "participants with", "subjects with",
              "diagnosed with", "confirmed", "aged", "years of age"]
_EXCL_CUES = ["exclusion criteria", "were excluded", "excluded if", "not eligible",
              "contraindication", "were ineligible"]
_ELIG_CUES = _INCL_CUES + _EXCL_CUES

_TABLE_ROW = re.compile(r"\d+\s*\(\s*\d+(?:\.\d+)?\s*%\s*\)")
_MANY_NUMS = re.compile(r"(?:\b\d+(?:\.\d+)?\b[^A-Za-z]{0,4}){5,}")
_CAPTION = re.compile(r"\b(figure|fig\.?|table|consort|supplement|appendix)\b", re.I)
_GLOSSARY = re.compile(r"\b[A-Z]{2,5}\s*=\s*[A-Za-z]")
_NCT = re.compile(r"NCT\d{6,8}", re.I)
_DATERANGE = re.compile(r"(?:from\s+)?(?:january|february|march|april|may|june|july|"
                        r"august|september|october|november|december)\s+\d{4}"
                        r"\s*(?:to|-|–|and|until)\s*(?:january|february|march|april|may|"
                        r"june|july|august|september|october|november|december)?\s*\d{4}", re.I)
_ENROLL = re.compile(r"\b(?:enrolled|recruited|included)\s+\d+\s+(?:patients|participants|subjects)", re.I)


def _strip_noise(s):
    s = _NCT.sub("", s)
    s = _DATERANGE.sub("", s)
    s = _ENROLL.sub("", s)
    s = re.sub(r"\(\s*\)", "", s)
    return clean_text(s)


def _looks_like_junk(sent):
    s = sent.strip()
    if len(s) < 25:
        return True
    if _TABLE_ROW.search(s) or _MANY_NUMS.search(s) or _GLOSSARY.search(s):
        return True
    if _CAPTION.search(s) and not any(c in s.lower() for c in _ELIG_CUES):
        return True
    if sum(ch.isdigit() for ch in s) > 0.30 * len(s):
        return True
    return False


def _eligibility_sentences(doc):
    blocks = []
    for sec in doc["sections"]:
        t = sec["title"].lower()
        if any(k in t for k in ["eligib", "inclusion", "exclusion", "criteria",
                                "patient", "participant", "subject", "population",
                                "method", "study design", "recruit", "enrol"]):
            blocks.append(" ".join(sec["paragraphs"]))
    if doc["abstract"]:
        blocks.append(doc["abstract"])
    if not blocks:
        blocks = [doc["full_text"][:6000]]
    incl, excl, seen = [], [], set()
    for block in blocks:
        for s in sent_split(block):
            sl = s.lower()
            if _looks_like_junk(s):
                continue
            key = sl[:80]
            if key in seen:
                continue
            if any(c in sl for c in _EXCL_CUES):
                seen.add(key)
                excl.append(_strip_noise(s))
            elif any(c in sl for c in _INCL_CUES):
                seen.add(key)
                incl.append(_strip_noise(s))
    incl = [s for s in incl if len(s) >= 20]
    excl = [s for s in excl if len(s) >= 20]
    return incl, excl


# LAYOUT-AWARE additions ------------------------------------------------------
# The enrollment/population sentence (usually in Methods) matches the CT.gov gold
# closely: "consecutive patients with X >=18 referred for...". Target it directly.
_ENROLL_SENT = re.compile(
    r"\b(consecutive patients|patients were (?:enrolled|recruited|included|eligible)|"
    r"we (?:enrolled|recruited|included|prospectively)|participants were "
    r"(?:enrolled|recruited|included)|were prospectively (?:enrolled|recruited)|"
    r"subjects were (?:enrolled|recruited|included)|were included if|"
    r"eligible (?:patients|participants|subjects)|inclusion criteria (?:were|included))\b",
    re.I)


def _enrollment_sentences(doc):
    """Pull the Methods population/enrollment sentence(s) that describe WHO was in
    the study — the closest paper-text analog to CT.gov inclusion criteria."""
    incl, excl, seen = [], [], set()
    method_blocks = []
    for sec in doc["sections"]:
        t = sec["title"].lower()
        if any(k in t for k in ["method", "material", "patient", "participant",
                                "subject", "population", "study design", "recruit",
                                "enrol", "eligib", "inclusion", "exclusion", "cohort"]):
            method_blocks.append(" ".join(sec["paragraphs"]))
    if not method_blocks and doc["abstract"]:
        method_blocks = [doc["abstract"]]
    for block in method_blocks:
        for s in sent_split(block):
            if _looks_like_junk(s):
                continue
            sl = s.lower()
            key = sl[:80]
            if key in seen:
                continue
            if any(c in sl for c in _EXCL_CUES):
                seen.add(key)
                excl.append(_strip_noise(s))
            elif _ENROLL_SENT.search(s) or any(c in sl for c in _INCL_CUES):
                seen.add(key)
                incl.append(_strip_noise(s))
    incl = [s for s in incl if len(s) >= 20]
    excl = [s for s in excl if len(s) >= 20]
    return incl, excl


def _elig_from_list_items(doc):
    """If the paper has explicit bulleted list-items that look like criteria,
    use them — they are the cleanest match to the gold's bulleted format."""
    items = doc.get("list_items", [])
    if not items:
        return [], []
    incl, excl = [], []
    for it in items:
        il = it.lower()
        if _looks_like_junk(it):
            continue
        if any(c in il for c in _EXCL_CUES) or "exclu" in il:
            excl.append(_strip_noise(it))
        elif any(c in il for c in _INCL_CUES) or "inclu" in il or "eligib" in il:
            incl.append(_strip_noise(it))
    return incl, excl


def extract_eligibility(doc, cap, mode="base"):
    """mode='base'  -> original sentence selection
       mode='layout'-> list-items first, then Methods enrollment sentences,
                       then fall back to base. Chosen by holdout A/B below."""
    if mode == "layout":
        incl, excl = _elig_from_list_items(doc)
        if not incl and not excl:
            incl, excl = _enrollment_sentences(doc)
        if not incl and not excl:
            incl, excl = _eligibility_sentences(doc)
    else:
        incl, excl = _eligibility_sentences(doc)

    parts = []
    if incl:
        parts.append("Inclusion Criteria: " + " ".join(incl))
    if excl:
        parts.append("Exclusion Criteria: " + " ".join(excl))
    text = clean_text(" ".join(parts))
    if not text:
        abs_sents = [_strip_noise(s) for s in sent_split(doc["abstract"]) if not _looks_like_junk(s)]
        text = clean_text(" ".join(abs_sents[:3])) or "Not Specified"
    return text[:cap] if text != "Not Specified" else text


# ============================================================================
# SECTION 5 — AGE (read from eligibility; else 18/NotSpecified; never guess max)
# ============================================================================
def extract_age(elig_text, doc):
    t = clean_text(elig_text).lower()
    for pat in [r"(?:aged|ages|age)\s*(?:between\s*)?(\d{1,3})\s*(?:-|–|to|and)\s*(\d{1,3})",
                r"between\s*(\d{1,3})\s*and\s*(\d{1,3})\s*(?:years|year|yrs|yr)"]:
        m = re.search(pat, t)
        if m:
            a, b = int(m.group(1)), int(m.group(2))
            if 1 <= a <= 120 and 1 <= b <= 120 and a < b:
                return f"{min(a,b)} Years", f"{max(a,b)} Years"

    min_age, max_age = None, None
    m = re.search(r"(?:≥|>=|>|older than|at least|aged?\s*over|minimum age of)\s*(\d{1,3})", t)
    if m and 1 <= int(m.group(1)) <= 120:
        min_age = f"{int(m.group(1))} Years"
    else:
        m = re.search(r"(?:age|aged)\s*(\d{1,3})\s*(?:years?)?\s*(?:old\s*)?"
                      r"(?:or (?:more|older|above)|and (?:older|above))", t)
        if m and 1 <= int(m.group(1)) <= 120:
            min_age = f"{int(m.group(1))} Years"

    m = re.search(r"(?:≤|<=|<|younger than|no older than|up to|maximum age of|aged?\s*under)\s*(\d{1,3})", t)
    if m and 1 <= int(m.group(1)) <= 120:
        max_age = f"{int(m.group(1))} Years"

    if min_age is None and re.search(r"\badults?\b", t):
        min_age = "18 Years"
    if min_age is None:
        min_age = AGE_MIN_DEFAULT
    if max_age is None:
        max_age = AGE_MAX_DEFAULT
    return min_age, max_age


# ============================================================================
# SECTION 6 — SEX (inclusion-context only)
# ============================================================================
def extract_sex(doc):
    incl, _ = _eligibility_sentences(doc)
    incl_text = " ".join(incl).lower()
    title_abs = clean_text(" ".join([doc["title"], doc["abstract"]])).lower()
    female_incl = re.search(r"\b(women only|females only|female patients? were (?:eligible|included|recruited)|"
                            r"only (?:women|female)|postmenopausal women were|"
                            r"pregnant women were (?:eligible|included|recruited))\b", incl_text)
    male_incl = re.search(r"\b(men only|males only|male patients? were (?:eligible|included|recruited)|"
                          r"only (?:men|male))\b", incl_text)
    female_disease = re.search(r"\b(cervical cancer|ovarian cancer|endometrial|gestational|"
                               r"pregnan|breast cancer in women)\b", title_abs)
    male_disease = re.search(r"\b(prostate cancer|prostatic|testicular|erectile)\b", title_abs)
    if male_incl or male_disease:
        return "MALE"
    if female_incl or female_disease:
        return "FEMALE"
    return "ALL"


# ============================================================================
# SECTION 7 — CONDITIONS (title/abstract-focused; wrong-match fix)
# ============================================================================
_GENERIC_TERMS = {"cancer", "disease", "diseases", "tumor", "tumour", "neoplasm",
                  "neoplasms", "disorder", "disorders", "syndrome", "condition"}

_VERB_MARKERS = {"is", "are", "was", "were", "be", "can", "could", "may", "might",
                 "detect", "detects", "detected", "altered", "cause", "causes",
                 "caused", "show", "shows", "shown", "found", "hallmark", "feature",
                 "associated", "reported", "observed", "measured"}

DISEASE_HINTS = ["cancer", "carcinoma", "tumor", "tumour", "neoplasm", "lymphoma",
                 "leukemia", "leukaemia", "melanoma", "sarcoma", "metastas", "asthma",
                 "embolism", "stroke", "hemorrhage", "haemorrhage", "infection",
                 "syndrome", "disease", "disorder", "failure", "diabetes",
                 "hypertension", "depression", "dementia", "arthritis", "fibrosis",
                 "pneumonia", "covid", "infarction", "ischemia", "ischaemia",
                 "sarcoidosis", "aspergillosis", "pancreatitis", "hernia", "herniation",
                 "epilepsy", "obesity", "fracture", "sclerosis", "sepsis", "aneurysm",
                 "cholangiocarcinoma", "adenocarcinoma", "glioma", "glioblastoma",
                 "cardiomyopathy", "angiopathy", "hyperplasia"]

BAD_CONDITION_TERMS = {"patient", "patients", "treatment", "therapy", "diagnosis",
                       "diagnostic", "healthy", "healthy volunteer", "healthy volunteers",
                       "control", "controls", "mri", "magnetic resonance imaging",
                       "computed tomography", "ct", "ultrasonography", "ultrasound",
                       "imaging", "artificial intelligence", "learning", "behavior",
                       "comfort", "stress", "prognostic factors", "echocardiography",
                       "study", "studies"}

_ALLOWED_SINGLE = {"stroke", "cancer", "dementia", "obesity", "asthma", "diabetes",
                   "hypertension", "depression", "epilepsy", "sepsis", "pneumonia",
                   "sarcoidosis", "melanoma", "glioblastoma", "leukemia", "leukaemia",
                   "hernia", "aneurysm", "fibrosis", "arthritis", "osteoarthritis"}

_APOS_FIX = re.compile(r"\bS\s+(Disease|Syndrome|Sarcoma|Lymphoma)\b")

_DISEASE_HEAD = (r"cancers?|carcinomas?|tumou?rs?|neoplasms?|lymphomas?|leukae?mias?|"
    r"sarcomas?|melanomas?|sarcoidosis|aspergillosis|cardiomyopath(?:y|ies)|"
    r"angiopath(?:y|ies)|hyperplasias?|diseases?|syndromes?|disorders?|infections?|"
    r"embolisms?|strokes?|h[ae]morrhages?|failures?|diabetes|hypertension|"
    r"dementias?|depression|arthritis|fibrosis|pneumonias?|pancreatitis|hernias?|"
    r"epilepsy|sclerosis|cirrhosis|hepatitis|cysts?|malignanc(?:y|ies)|lesions?|"
    r"metastas[ei]s|injur(?:y|ies)|edema|oedema|hydrocephalus|infarctions?|"
    r"isch[ae]mias?|aneurysms?|fractures?|sepsis|stenosis|thrombosis|nodules?|"
    r"polyps?|adenomas?|gliomas?|glioblastomas?|cholangiocarcinomas?|"
    r"adenocarcinomas?|osteoarthritis|obesity|asthma")

_CAND_RE = re.compile(
    rf"\b([A-Za-z][A-Za-z\-]*(?:\s+[A-Za-z\-]+){{0,3}}\s*(?:{_DISEASE_HEAD}))\b", re.I)

_FILLER_RE = re.compile(
    r"^(patients?|adults?|subjects?|participants?|with|without|suspected|known|"
    r"among|of|the|a|an|and|or|diagnosis|diagnosed|for|in|has|have|had|clinically|"
    r"isolated|primary|secondary|newly|both|either|this|that|these|those|"
    r"early|late|stage|type|our|study|group|control)\s+", re.I)


def _repair_condition_name(name):
    name = clean_text(name)
    name = _APOS_FIX.sub(r"\1", name).strip()
    name = re.sub(r"^[\s'\"\-]+", "", name)
    return name


def _contains_verb(phrase):
    return bool(set(tokenise(phrase)) & _VERB_MARKERS)


def is_condition(x):
    x = _repair_condition_name(x)
    xl = x.lower()
    if len(x) < 4 or xl in BAD_CONDITION_TERMS:
        return False
    if _contains_verb(xl) or len(xl.split()) > 6:
        return False
    if any(b in xl for b in ["assessment", "parameter", "purpose ", "relevance statement", "hallmark"]):
        return False
    if xl in _ALLOWED_SINGLE:
        return True
    return any(h in xl for h in DISEASE_HINTS)


def _titlecase_condition(s):
    s = _repair_condition_name(s)
    return " ".join(w if (w.isupper() and len(w) <= 4) else w.capitalize() for w in s.split())


def _clean_disease_name(phrase):
    p = phrase.strip()
    prev = None
    while prev != p:
        prev = p
        p = _FILLER_RE.sub("", p).strip()
    words = p.split()
    if len(words) > 3:
        p = " ".join(words[-3:])
    return p


def _dedupe_drop_generic(items):
    cleaned = []
    for it in items:
        it = _repair_condition_name(it)
        if is_condition(it):
            cleaned.append(it)
    cleaned = list(dict.fromkeys(cleaned))
    specific = [c for c in cleaned if c.lower() not in _GENERIC_TERMS]
    return (specific if specific else cleaned)[:4]


def condition_string(items):
    out = _dedupe_drop_generic(items)
    if not out:
        out = ["Disease"]
    return repr(out) if FORMAT_AS_LIST else ", ".join(out)


def scispacy_diseases(doc):
    if _SCISPACY is None:
        return []
    text = clean_text(" ".join([doc["title"], doc["abstract"], doc["keywords"]]))[:5000]
    if not text:
        return []
    try:
        ents = _SCISPACY(text).ents
    except Exception:
        return []
    found, seen = [], set()
    for e in ents:
        if e.label_ == "DISEASE":
            name = _titlecase_condition(e.text)
            k = name.lower()
            if k and k not in seen and 4 <= len(name) <= 60 and not _contains_verb(k):
                seen.add(k)
                found.append(name)
    tl = text.lower()
    found.sort(key=lambda n: tl.count(n.lower()), reverse=True)
    return found[:4]


def _diseases_from_title_abstract(doc):
    """Match diseases in title+abstract+keywords ONLY -> the central disease,
    not a stray body word. This is the lymphoma->Stroke wrong-match fix."""
    ta = clean_text(" ".join([doc["title"], doc["abstract"], doc["keywords"]]))
    if not ta:
        return []
    cleaned, seen = [], set()
    for h in _CAND_RE.findall(ta):
        c = _clean_disease_name(h)
        cl = c.lower()
        if not cl or len(c) < 4 or cl in seen:
            continue
        if cl in BAD_CONDITION_TERMS or _contains_verb(cl):
            continue
        seen.add(cl)
        cleaned.append(c.title())
    tl = ta.lower()
    cleaned.sort(key=lambda n: tl.count(n.lower()), reverse=True)
    specific = [c for c in cleaned if c.lower() not in _GENERIC_TERMS]
    return (specific if specific else cleaned)[:4]


def build_condition_vocab(train_part):
    counts = Counter()
    for xs in train_part["conditions"].apply(parse_list_label):
        for c in xs:
            c = _repair_condition_name(c)
            if is_condition(c):
                counts[c] += 1
    vocab = []
    for term in counts:
        if term.lower() in {"disease", "diseases", "neoplasm", "neoplasms", "disorder",
                            "disorders", "tumor", "tumour", "condition", "syndrome"}:
            continue
        if len(term) < 4:
            continue
        vocab.append(term)
    return sorted(set(vocab), key=len, reverse=True)


def extract_conditions(doc, vocab):
    # 1) scispaCy if available (optional bonus; harmless no-op if not installed)
    hits = scispacy_diseases(doc)
    if hits:
        return condition_string(hits)
    # 2) PROVEN WINNER (measured on holdout: 0.563 vs 0.507/0.502 alternatives):
    #    training vocab match in title+abstract+keywords ONLY, ranked by frequency,
    #    capped to top 3. This never emits messy multi-word fragments, which is
    #    why it beat the "clean extraction" and old title/abstract-regex methods.
    ta = clean_text(" ".join([doc["title"], doc["abstract"], doc["keywords"]])).lower()
    hits = [c for c in vocab if len(c) > 4 and c.lower() in ta]
    if hits:
        hits.sort(key=lambda n: ta.count(n.lower()), reverse=True)
        return condition_string(hits[:3])
    # 3) fallback: clean disease-name extraction from title+abstract (dedup by head word)
    hits = _diseases_from_title_abstract(doc)
    if hits:
        return condition_string(hits)
    # 4) last resort: any disease-hint word in the body
    body = clean_text(doc["full_text"][:3000]).lower()
    for h in DISEASE_HINTS:
        m = re.search(rf"\b([A-Za-z][A-Za-z\-]*\s+)?{h}\w*\b", body)
        if m:
            cand = _clean_disease_name(m.group(0)).title()
            if is_condition(cand):
                return condition_string([cand])
    return condition_string([])


# ============================================================================
# SECTION 8 — STUDY TYPE
# ============================================================================
def study_type_prior(train_part):
    vc = train_part["study_type"].fillna("Not Specified").astype(str).str.upper().value_counts()
    print("\n[study_type training prior]\n", vc)
    return (vc.index[0] if len(vc) else "OBSERVATIONAL"), vc


_INTER_CUES = ["randomized", "randomised", "randomly assigned", "double-blind",
               "single-blind", "placebo", "treatment arm", "intervention group",
               "phase ii", "phase iii", "phase 2", "phase 3", "controlled trial",
               "assigned to receive", "allocated to"]
_OBS_CUES = ["retrospective", "prospective cohort", "observational", "cross-sectional",
             "case-control", "registry", "chart review", "medical records",
             "cohort study", "consecutive patients", "medical record review"]


def study_type_predict(doc, default_label):
    t = clean_text(" ".join([doc["title"], doc["abstract"], doc["full_text"][:4000]])).lower()
    inter = sum(c in t for c in _INTER_CUES)
    obs = sum(c in t for c in _OBS_CUES)
    strong = any(c in t for c in ["randomized", "randomised", "placebo", "double-blind",
                                  "phase ii", "phase iii", "controlled trial"])
    if strong and inter >= 2 and inter > obs:
        return "INTERVENTIONAL"
    if obs >= 1 and obs >= inter:
        return "OBSERVATIONAL"
    if inter > obs and strong:
        return "INTERVENTIONAL"
    return default_label


# ============================================================================
# SECTION 9 — Prediction assembly
# ============================================================================
def predict_frame(df, docs, vocab, study_default, elig_cap, elig_mode="base"):
    rows = []
    for _, r in df.iterrows():
        doc = docs[r["pmcid_clean"]]
        elig = extract_eligibility(doc, elig_cap, mode=elig_mode)
        src = elig if elig != "Not Specified" else doc["full_text"][:3000]
        mn, mx = extract_age(src, doc)
        rows.append({
            "pmcids": clean_pmcid(r["pmcids"]),
            "conditions": extract_conditions(doc, vocab),
            "study_type": study_type_predict(doc, study_default),
            "sex": extract_sex(doc),
            "minimum_age": mn,
            "maximum_age": mx,
            "eligibility_criteria": elig,
        })
    return pd.DataFrame(rows)


def gold_frame(part):
    return pd.DataFrame({
        "conditions": part["conditions"].apply(
            lambda x: repr(_dedupe_drop_generic(parse_list_label(x))) if FORMAT_AS_LIST
            else ", ".join(_dedupe_drop_generic(parse_list_label(x)))),
        "study_type": part["study_type"].fillna("Not Specified").astype(str).str.upper(),
        "sex": (part["sex"].fillna("ALL").astype(str).str.upper()
                if "sex" in part else pd.Series(["ALL"] * len(part))),
        "minimum_age": (part["minimum_age"].fillna("Not Specified").astype(str)
                        if "minimum_age" in part else pd.Series(["Not Specified"] * len(part))),
        "maximum_age": (part["maximum_age"].fillna("Not Specified").astype(str)
                        if "maximum_age" in part else pd.Series(["Not Specified"] * len(part))),
        "eligibility_criteria": part["eligibility_criteria"].fillna("Not Specified").astype(str),
    }).reset_index(drop=True)


# ============================================================================
# SECTION 10 — LOCAL VALIDATION (per-field, so you see each fix)
# ============================================================================
from sklearn.model_selection import train_test_split  # noqa: E402

tr_idx, va_idx = train_test_split(np.arange(len(train_df)), test_size=0.22,
                                  random_state=42, stratify=train_df["study_type"].fillna("NA"))
tr = train_df.iloc[tr_idx].reset_index(drop=True)
va = train_df.iloc[va_idx].reset_index(drop=True)

vocab_tr = build_condition_vocab(tr)
study_default_tr, _ = study_type_prior(tr)
va_gold = gold_frame(va)

print("\n" + "=" * 70)
print("ELIGIBILITY: A/B test extraction MODE x cap sweep (holdout)")
print("=" * 70)
best_cap, best_score, ELIG_MODE = ELIG_CAP_CANDIDATES[0], -1.0, "base"
for mode in ["base", "layout"]:
    print(f"\n-- mode = {mode} --")
    for cap in ELIG_CAP_CANDIDATES:
        vp = predict_frame(va, train_docs, vocab_tr, study_default_tr, cap, elig_mode=mode)
        total, br = score_frame(vp, va_gold)
        print(f"cap={cap:>4} | composite={total:.4f} | elig={br['elig']:.3f} "
              f"cond={br['cond']:.3f} study={br['study']:.3f} num={br['num']:.3f}")
        if total > best_score:
            best_score, best_cap, ELIG_MODE = total, cap, mode
print(f"\n>>> Best: mode={ELIG_MODE}, cap={best_cap} (composite={best_score:.4f})")

vp_best = predict_frame(va, train_docs, vocab_tr, study_default_tr, best_cap, elig_mode=ELIG_MODE)
print("\nstudy_type exact-match:",
      round((vp_best["study_type"].values == va_gold["study_type"].values).mean(), 3))
print("sex exact-match:",
      round((vp_best["sex"].values == va_gold["sex"].values).mean(), 3))
print("min_age exact-match:",
      round((vp_best["minimum_age"].values == va_gold["minimum_age"].values).mean(), 3))
print("max_age exact-match:",
      round((vp_best["maximum_age"].values == va_gold["maximum_age"].values).mean(), 3))
print("['Disease'] fallback:",
      int(vp_best["conditions"].isin(["['Disease']", "Disease"]).sum()), "/", len(vp_best))



# ============================================================================
# SECTION 9 — CONSERVATIVE AGE FIXES  (root-cause diagnostic showed these)
# ============================================================================
# Diagnosed failure patterns on unresolved papers:
#   - we parsed RESULT statistics ("median age 69 years") as if they were
#     eligibility criteria  -> must ignore
#   - we parsed CASE-REPORT ages ("a 45-year-old female presented") as criteria
#     -> must ignore
#   - we parsed ANIMAL ages ("mice 5 weeks of age") as human ages
#     -> animal studies should be Not Specified
# These fixes mostly stop us being WRONG. On the number metric one wrong number
# scores 0 for that row, so being conservative genuinely helps.
# ============================================================================
_RESULT_STAT = re.compile(
    r"\b(median|mean|average)\s+age\b|\bage[d]?\s*[:,]?\s*\d+(\.\d+)?\s*[\u00b1+]/?-|"
    r"\b(sd|iqr|range)\s*[:=]|\bmean\s*\u00b1|\b\d+\s*\u00b1\s*\d+", re.I)
_CASE_REPORT = re.compile(
    r"\ba\s+\d{1,3}[-\s]year[-\s]old\b|\ban?\s+\d{1,3}[-\s]y[-\s]?o\b|"
    r"\b\d{1,3}[-\s]year[-\s]old\s+(man|woman|male|female|boy|girl|patient)\b.*"
    r"\b(presented|referred|admitted|reported)\b", re.I)
_ANIMAL = re.compile(
    r"\b(mice|mouse|murine|rat|rats|rodent|porcine|swine|canine|dog[s]?|"
    r"rabbit[s]?|primate[s]?|macaque|zebrafish|in\s+vivo\s+animal)\b", re.I)


def _is_animal_study(doc):
    """Animal papers should not report human ages."""
    head = (doc["title"] + " " + doc["abstract"]).lower()
    hits = len(_ANIMAL.findall(head))
    # require it to be prominent in title/abstract, not a passing mention
    return hits >= 2 or bool(re.search(r"\b(mice|mouse|rats?|murine)\b", doc["title"], re.I))


def _age_sentences_clean(text):
    """Drop sentences that are result-statistics or case descriptions."""
    keep = []
    for s in sent_split(text):
        if _RESULT_STAT.search(s):
            continue
        if _CASE_REPORT.search(s):
            continue
        keep.append(s)
    return " ".join(keep)


def extract_age_v2(elig_text, doc):
    """Conservative age extraction. Falls back to the proven defaults."""
    if _is_animal_study(doc):
        return AGE_MIN_DEFAULT, AGE_MAX_DEFAULT   # never trust animal ages
    cleaned = _age_sentences_clean(elig_text)
    if not cleaned.strip():
        cleaned = elig_text  # if we filtered everything, fall back
    return extract_age(cleaned, doc)


# ============================================================================
# SECTION 10 — CANDIDATE PASSAGE GENERATION
# ============================================================================
# The host said the task is to "obtain the most similar piece from the NXML
# file". So: chop each paper into candidate passages, then rank them.
# ============================================================================
MAX_CANDIDATES = 40           # per paper (CPU budget)
WINDOW_SIZES = (1, 2, 3)      # single sentences, and 2-3 sentence windows

_ELIG_SECTION_CUES = ["eligib", "inclusion", "exclusion", "criteria", "patient",
                      "participant", "subject", "population", "method", "material",
                      "study design", "recruit", "enrol", "cohort", "sample"]

_JUNKY = re.compile(
    r"^\s*(table|figure|fig\.|supplementary|appendix|abbreviation)", re.I)


def _candidate_blocks(doc):
    """Text blocks likely to contain eligibility info, in priority order."""
    blocks = []
    for sec in doc["sections"]:
        t = sec["title"].lower()
        if any(k in t for k in _ELIG_SECTION_CUES):
            for para in sec["paragraphs"]:
                blocks.append((sec["title"], para))
    if doc["abstract"]:
        blocks.append(("Abstract", doc["abstract"]))
    for it in doc.get("list_items", []):
        blocks.append(("List", it))
    if not blocks:
        blocks = [("Body", doc["full_text"][:6000])]
    return blocks


def generate_candidates(doc):
    """Return a list of dicts: {text, section, from_abstract, from_list, pos}."""
    cands, seen = [], set()
    blocks = _candidate_blocks(doc)
    total = max(1, len(blocks))
    for bi, (sec_title, block) in enumerate(blocks):
        sents = [s for s in sent_split(block) if len(s) >= 25 and not _JUNKY.match(s)]
        if not sents:
            continue
        for w in WINDOW_SIZES:
            for i in range(0, len(sents) - w + 1):
                txt = " ".join(sents[i:i + w])
                if len(txt) < 30 or len(txt) > 1200:
                    continue
                key = txt[:90].lower()
                if key in seen:
                    continue
                seen.add(key)
                cands.append({
                    "text": _strip_noise(txt),
                    "section": sec_title,
                    "from_abstract": sec_title == "Abstract",
                    "from_list": sec_title == "List",
                    "pos": bi / total,
                })
    # keep the most promising ones if we have too many (cheap prefilter by cues)
    if len(cands) > MAX_CANDIDATES:
        def cue_score(c):
            t = c["text"].lower()
            s = 0
            s += 3 * sum(k in t for k in ("inclusion criteria", "exclusion criteria",
                                          "eligib", "were included", "were excluded"))
            s += 2 * sum(k in t for k in ("enrolled", "recruited", "consecutive",
                                          "were eligible", "met the following"))
            s += 1 * bool(re.search(r"\b\d{1,3}\s*(?:years|yrs)\b", t))
            s += 1 * ("patients" in t or "participants" in t or "subjects" in t)
            s += 1 * c["from_list"]
            return s
        cands.sort(key=cue_score, reverse=True)
        cands = cands[:MAX_CANDIDATES]
    return cands


# ============================================================================
# SECTION 11 — LEARNED ELIGIBILITY RANKER  (the main new idea)
# ============================================================================
# For every TRAINING paper we know the gold. So for each candidate passage we
# can compute its TRUE FM3S score against that gold. That gives us labelled
# data: (features of candidate) -> (how good it actually is).
# We fit a small model to predict that score, then at TEST time we rank the
# candidates and keep the best ones. All data used is provided by the competition.
# ============================================================================
INCL_CUE_RE = re.compile(
    r"\b(inclusion criteria|were included|was included|eligible|eligibility|"
    r"were enrolled|were recruited|consecutive patients|we enrolled|we recruited|"
    r"met the following|had to be|required to)\b", re.I)
EXCL_CUE_RE = re.compile(
    r"\b(exclusion criteria|were excluded|was excluded|excluded if|"
    r"not eligible|ineligible|contraindicat)\b", re.I)
AGE_RE = re.compile(r"\b\d{1,3}\s*(?:years|yrs|y\.o\.|year-old)\b", re.I)
CLIN_RE = re.compile(
    r"\b(diagnos|confirmed|histolog|biopsy|stage|prior|previous|treatment|therapy|"
    r"history of|documented|underwent|referred)\b", re.I)
NEG_RE = re.compile(
    r"\b(pregnan|lactat|unable|refus|contraindicat|allerg|renal failure|"
    r"severe|comorbid|declined|withdrew)\b", re.I)
MODAL_RE = re.compile(r"\b(must|should|required|had to|shall|need to)\b", re.I)


def candidate_features(c, proto_sims):
    """Cheap, robust features. proto_sims = BioBERT sims to gold prototypes."""
    t = c["text"]
    tl = t.lower()
    n_words = max(1, len(t.split()))
    return [
        proto_sims.get("centroid", 0.0),
        proto_sims.get("best_cluster", 0.0),
        1.0 if INCL_CUE_RE.search(tl) else 0.0,
        1.0 if EXCL_CUE_RE.search(tl) else 0.0,
        1.0 if AGE_RE.search(tl) else 0.0,
        len(CLIN_RE.findall(tl)) / n_words * 10.0,
        len(NEG_RE.findall(tl)) / n_words * 10.0,
        1.0 if MODAL_RE.search(tl) else 0.0,
        1.0 if c["from_abstract"] else 0.0,
        1.0 if c["from_list"] else 0.0,
        1.0 if any(k in c["section"].lower() for k in ("method", "patient", "eligib",
                                                       "inclusion", "population")) else 0.0,
        min(len(t), 1200) / 1200.0,
        c["pos"],
        1.0 if ("patients" in tl or "participants" in tl or "subjects" in tl) else 0.0,
    ]


N_FEATURES = 14


def build_gold_prototypes(train_part, n_clusters=6):
    """Embed the training gold eligibility texts -> centroid + cluster centres.
    This teaches the ranker what a real criteria block 'looks like'."""
    golds = [clean_text(g) for g in train_part["eligibility_criteria"] if clean_text(g)]
    golds = [g for g in golds if len(g) > 40]
    if not golds or not _BERT.ok:
        return None
    try:
        embs = _BERT.model.encode(golds[:400], convert_to_numpy=True,
                                  normalize_embeddings=True, show_progress_bar=False,
                                  batch_size=32)
    except Exception as e:
        print(f"[warn] prototype embedding failed: {e!r}")
        return None
    centroid = embs.mean(axis=0)
    centroid = centroid / (np.linalg.norm(centroid) + 1e-9)
    clusters = None
    try:
        from sklearn.cluster import KMeans
        k = min(n_clusters, max(2, len(embs) // 20))
        km = KMeans(n_clusters=k, n_init=4, random_state=42).fit(embs)
        clusters = km.cluster_centers_
        clusters = clusters / (np.linalg.norm(clusters, axis=1, keepdims=True) + 1e-9)
    except Exception:
        pass
    return {"centroid": centroid, "clusters": clusters}


def embed_candidates(cands):
    """Embed candidate texts once (batched) -> numpy array."""
    if not cands or not _BERT.ok:
        return None
    try:
        return _BERT.model.encode([c["text"] for c in cands], convert_to_numpy=True,
                                  normalize_embeddings=True, show_progress_bar=False,
                                  batch_size=32)
    except Exception:
        return None


def proto_sims_for(embs, protos, i):
    if embs is None or protos is None:
        return {"centroid": 0.0, "best_cluster": 0.0}
    v = embs[i]
    out = {"centroid": float(np.dot(v, protos["centroid"]))}
    if protos["clusters"] is not None:
        out["best_cluster"] = float(np.max(protos["clusters"] @ v))
    else:
        out["best_cluster"] = out["centroid"]
    return out


def train_eligibility_ranker(train_part, docs, protos, n_papers=180, verbose=True):
    """Label candidates with their TRUE FM3S vs gold, then fit a small regressor."""
    if not _BERT.ok:
        print("[ranker] BioBERT unavailable -> learned ranker disabled.")
        return None
    sub = train_part.head(n_papers)
    X, y = [], []
    if verbose:
        print(f"[ranker] building training signal from {len(sub)} papers "
              f"(computing true FM3S per candidate) ...")
    for n, (_, r) in enumerate(sub.iterrows()):
        pid = r["pmcid_clean"]
        doc = docs.get(pid)
        if doc is None:
            continue
        gold = clean_text(r["eligibility_criteria"])
        if len(gold) < 40:
            continue
        cands = generate_candidates(doc)
        if not cands:
            continue
        embs = embed_candidates(cands)
        for i, c in enumerate(cands):
            ps = proto_sims_for(embs, protos, i)
            try:
                score = fm3s_similarity(c["text"], gold)   # the REAL metric
            except Exception:
                continue
            X.append(candidate_features(c, ps))
            y.append(float(score))
        if verbose and (n + 1) % 30 == 0:
            print(f"  ... {n + 1}/{len(sub)} papers, {len(X)} labelled candidates")
    if len(X) < 200:
        print(f"[ranker] too few training samples ({len(X)}) -> ranker disabled.")
        return None
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    try:
        from sklearn.ensemble import GradientBoostingRegressor
        model = GradientBoostingRegressor(n_estimators=160, max_depth=3,
                                          learning_rate=0.07, subsample=0.9,
                                          random_state=42)
        model.fit(X, y)
        if verbose:
            print(f"[ranker] trained on {len(X)} candidates "
                  f"(target = true FM3S vs gold). train R^2={model.score(X, y):.3f}")
        return model
    except Exception as e:
        print(f"[ranker] sklearn failed ({e!r}); trying Ridge.")
        try:
            from sklearn.linear_model import Ridge
            model = Ridge(alpha=1.0).fit(X, y)
            return model
        except Exception as e2:
            print(f"[ranker] disabled: {e2!r}")
            return None


def eligibility_learned(doc, ranker, protos, top_k, cap):
    """Rank candidates with the learned model, keep the best, assemble."""
    cands = generate_candidates(doc)
    if not cands or ranker is None:
        return None
    embs = embed_candidates(cands)
    feats = [candidate_features(c, proto_sims_for(embs, protos, i))
             for i, c in enumerate(cands)]
    try:
        scores = ranker.predict(np.asarray(feats, dtype=float))
    except Exception:
        return None
    order = np.argsort(scores)[::-1][:top_k]
    chosen = [cands[i] for i in order]
    # split into inclusion-like vs exclusion-like, mirroring the gold's shape
    incl, excl = [], []
    for c in chosen:
        (excl if EXCL_CUE_RE.search(c["text"]) else incl).append(c["text"])
    parts = []
    if incl:
        parts.append("Inclusion Criteria: " + " ".join(incl))
    if excl:
        parts.append("Exclusion Criteria: " + " ".join(excl))
    text = clean_text(" ".join(parts))
    if not text:
        return None
    return text[:cap]


# ============================================================================
# SECTION 12 — PREDICTION ASSEMBLY  (mode-switchable, so we can A/B safely)
# ============================================================================
def predict_frame_v10(df, docs, vocab, study_default, elig_cap,
                      elig_mode="base", ranker=None, protos=None, top_k=6,
                      age_mode="v2"):
    rows = []
    for _, r in df.iterrows():
        doc = docs[r["pmcid_clean"]]

        elig = None
        if elig_mode == "learned":
            elig = eligibility_learned(doc, ranker, protos, top_k, elig_cap)
        if elig is None:                       # base method, or learned failed
            elig = extract_eligibility(doc, elig_cap, mode="base")

        src = elig if elig != "Not Specified" else doc["full_text"][:3000]
        if age_mode == "v2":
            mn, mx = extract_age_v2(src, doc)
        else:
            mn, mx = extract_age(src, doc)

        rows.append({
            "pmcids": clean_pmcid(r["pmcids"]),
            "conditions": extract_conditions(doc, vocab),
            "study_type": study_type_predict(doc, study_default),
            "sex": extract_sex(doc),
            "minimum_age": mn,
            "maximum_age": mx,
            "eligibility_criteria": elig,
        })
    return pd.DataFrame(rows)


# ============================================================================
# SECTION 13 — HOLDOUT VALIDATION + A/B  (safety guard on every change)
# ============================================================================
print("\n" + "=" * 74)
print("STEP 1 — build gold prototypes + train the eligibility ranker")
print("=" * 74)
protos = build_gold_prototypes(tr)
print(f"[proto] gold prototypes: {'ready' if protos else 'unavailable (BioBERT off)'}")
ranker = train_eligibility_ranker(tr, train_docs, protos, n_papers=RANKER_TRAIN_PAPERS)

print("\n" + "=" * 74)
print("STEP 2 — AGE: conservative fixes A/B (holdout)")
print("=" * 74)
vp_age_old = predict_frame_v10(va, train_docs, vocab_tr, study_default_tr,
                               best_cap, elig_mode="base", age_mode="v1")
vp_age_new = predict_frame_v10(va, train_docs, vocab_tr, study_default_tr,
                               best_cap, elig_mode="base", age_mode="v2")
tot_age_old, br_old = score_frame(vp_age_old, va_gold)
tot_age_new, br_new = score_frame(vp_age_new, va_gold)
print(f"age v1 (original):     composite={tot_age_old:.4f}  num={br_old['num']:.3f}")
print(f"age v2 (conservative): composite={tot_age_new:.4f}  num={br_new['num']:.3f}")
AGE_MODE = "v2" if tot_age_new >= tot_age_old else "v1"
print(f">>> AGE_MODE = {AGE_MODE}")

print("\n" + "=" * 74)
print("STEP 3 — ELIGIBILITY: learned ranker vs base rules (holdout)")
print("=" * 74)
base_total, base_br = score_frame(
    predict_frame_v10(va, train_docs, vocab_tr, study_default_tr, best_cap,
                      elig_mode="base", age_mode=AGE_MODE), va_gold)
print(f"BASE (rules)     composite={base_total:.4f}  elig={base_br['elig']:.3f}")

ELIG_MODE_V10, BEST_TOPK, BEST_CAP_V10 = "base", 6, best_cap
best_total = base_total

if ranker is not None:
    for top_k in TOPK_CANDIDATES:
        for cap in ELIG_CAP_CANDIDATES:
            vp = predict_frame_v10(va, train_docs, vocab_tr, study_default_tr, cap,
                                   elig_mode="learned", ranker=ranker, protos=protos,
                                   top_k=top_k, age_mode=AGE_MODE)
            tot, br = score_frame(vp, va_gold)
            print(f"LEARNED top_k={top_k:>2} cap={cap:>4} | composite={tot:.4f} "
                  f"elig={br['elig']:.3f} cond={br['cond']:.3f} "
                  f"study={br['study']:.3f} num={br['num']:.3f}")
            if tot > best_total:
                best_total, ELIG_MODE_V10, BEST_TOPK, BEST_CAP_V10 = tot, "learned", top_k, cap

print(f"\n>>> WINNER: mode={ELIG_MODE_V10}  top_k={BEST_TOPK}  cap={BEST_CAP_V10}  "
      f"composite={best_total:.4f}")
if ELIG_MODE_V10 == "base":
    print("    (learned ranker did NOT beat the rules -> base kept. Score is safe.)")
else:
    print(f"    (learned ranker WINS: {base_total:.4f} -> {best_total:.4f})")

print("\n" + "=" * 74)
print("FINAL HOLDOUT SCORE (rule-compliant, extraction-only)")
print("=" * 74)
vp_final = predict_frame_v10(va, train_docs, vocab_tr, study_default_tr, BEST_CAP_V10,
                             elig_mode=ELIG_MODE_V10, ranker=ranker, protos=protos,
                             top_k=BEST_TOPK, age_mode=AGE_MODE)
fin_total, fin_br = score_frame(vp_final, va_gold)
print(f"composite = {fin_total:.4f}")
print(f"  eligibility (50%) = {fin_br['elig']:.3f}")
print(f"  conditions  (20%) = {fin_br['cond']:.3f}")
print(f"  study_type  (15%) = {fin_br['study']:.3f}")
print(f"  age/number  (15%) = {fin_br['num']:.3f}")
_fb = int(vp_final["conditions"].isin(["['Disease']", "Disease"]).sum())
print("\n['Disease'] fallback rows:", _fb, "/", len(vp_final))


# ============================================================================
# SECTION 14 — REFIT ON ALL TRAINING DATA + PREDICT TEST + WRITE SUBMISSION
# ============================================================================
print("\n" + "=" * 74)
print("STEP 4 — refit on ALL training data, predict test, write submission.csv")
print("=" * 74)
vocab_full = build_condition_vocab(train_df)
study_default_full, _ = study_type_prior(train_df)

# retrain prototypes + ranker on the FULL training set for the final model
protos_full = build_gold_prototypes(train_df)
ranker_full = ranker
if ELIG_MODE_V10 == "learned":
    ranker_full = train_eligibility_ranker(train_df, train_docs, protos_full,
                                           n_papers=RANKER_TRAIN_PAPERS_FINAL)
    if ranker_full is None:
        print("[warn] final ranker failed to train -> falling back to base rules.")
        ELIG_MODE_V10 = "base"

# NOTE: the v10 test prediction + submission write is intentionally SKIPPED here.
# v11 (below) produces the final submission, so predicting test twice would just
# double the slowest step. We keep only the fitted objects v11 needs:
#   vocab_full, study_default_full, protos_full, ranker_full
print("[v10] models fitted on full training data; final submission is produced by v11 below.")


# ============================================================================
# SECTION 15 — LEARNED CONDITIONS RANKER   (v11: the biggest remaining gap)
# ============================================================================
# WHY: conditions scored 0.555 and 30% of test papers fell back to ['Disease'].
# Two root causes, both fixed here:
#   (a) MEASURED: 43% of gold "conditions" are NOT diseases -- they are
#       procedures / anatomy / imaging (SPECT-CT, VATS, Lung, PET). The old
#       extractor only looked for disease words, so it could never find those.
#       -> candidate generation now covers diseases AND procedures AND anatomy
#          AND imaging modalities AND the paper's own keywords.
#   (b) The old extractor used fixed rules. Here we LEARN the ranking: for every
#       candidate term we compute its TRUE BioBERT-cosine score against the gold
#       (the exact metric used for this field) and fit a model to predict it.
#       All learning uses provided training data only.
# The fallback string is also OPTIMISED on the training data instead of blindly
# using ['Disease'].
# ============================================================================

_DIS_HEAD = (r"cancers?|carcinomas?|tumou?rs?|neoplasms?|sarcomas?|lymphomas?|"
             r"leukae?mias?|melanomas?|metastas[ei]s|disease[s]?|disorder[s]?|"
             r"syndrome[s]?|injur(?:y|ies)|failure|infections?|deficienc(?:y|ies)|"
             r"stroke|diabetes|hypertension|obesity|dementia|arthritis|asthma|"
             r"sclerosis|epilepsy|depression|anxiety|pain|fractures?|hernias?|"
             r"ulcers?|itis|osis|emia|pathy|opathy|aemia|algia")
_DISEASE_PHRASE = re.compile(
    r"\b((?:[A-Za-z][A-Za-z\-']{2,}\s+){0,3}[A-Za-z\-']*(?:" + _DIS_HEAD + r"))\b", re.I)

# gold conditions are often procedures / imaging / anatomy -- these MUST be candidates
_PROC_IMG = re.compile(
    r"\b(PET(?:[/\-]CT|[/\-]MRI)?|SPECT(?:[/\-]CT)?|CT\s*scan|MRI|fMRI|MRS|"
    r"ultrasound|echocardiograph\w*|mammograph\w*|colonoscopy|endoscopy|biopsy|"
    r"VATS|laparoscop\w*|thoracoscop\w*|angiograph\w*|radiotherapy|chemotherapy|"
    r"immunotherapy|surgery|resection|transplant\w*|dialysis|screening|"
    r"vaccination|rehabilitation|physiotherapy)\b", re.I)
_ANATOMY = re.compile(
    r"\b(lung|breast|prostate|colon|rectal|colorectal|liver|hepatic|kidney|renal|"
    r"brain|cerebral|cardiac|heart|coronary|pulmonary|gastric|pancreatic|ovarian|"
    r"cervical|bladder|thyroid|skin|bone|spine|spinal|knee|hip|shoulder|"
    r"oesophageal|esophageal|head and neck)\b", re.I)

_COND_STOP = {"study", "trial", "patients", "patient", "analysis", "results",
              "methods", "background", "objective", "purpose", "conclusion",
              "using", "based", "novel", "effect", "effects", "role", "review",
              "randomized", "randomised", "controlled", "clinical", "outcomes"}


_LEAD_JUNK = re.compile(
    r"^(?:for|with|in|on|of|the|a|an|and|or|during|after|before|by|to|from|"
    r"suspected|combined|emerging|novel|using|via|among|between|towards?)\s+",
    re.I)


def _titlecase_term(t):
    t = re.sub(r"\s+", " ", t).strip(" ,.;:()[]-")
    # strip leading prepositions/articles so we get 'Sarcoidosis', not
    # 'For Suspected Cardiac Sarcoidosis' (keeps candidates clean for the ranker)
    prev = None
    while t and t != prev:
        prev = t
        t = _LEAD_JUNK.sub("", t).strip(" ,.;:()[]-")
    if not t:
        return ""
    words = t.split()
    out = []
    for w in words:
        if w.isupper() and len(w) <= 6:      # keep acronyms: PET, CT, VATS
            out.append(w)
        else:
            out.append(w[:1].upper() + w[1:].lower())
    return " ".join(out)


def generate_condition_candidates(doc, vocab):
    """All plausible condition terms: diseases, procedures, anatomy, imaging,
    the paper's own keywords, and training-vocabulary matches."""
    title = clean_text(doc["title"])
    abstract = clean_text(doc["abstract"])
    kwds = clean_text(doc["keywords"])
    ta = (title + " " + abstract).lower()
    body = doc["full_text"][:4000].lower()

    seen, cands = set(), []

    def add(term, source, in_title):
        t = _titlecase_term(term)
        if not t or len(t) < 3:
            return
        tl = t.lower()
        if tl in _COND_STOP or tl in seen:
            return
        if len(tl.split()) > 5:
            return
        seen.add(tl)
        cands.append({
            "term": t,
            "in_title": bool(in_title),
            "in_abstract": tl in abstract.lower(),
            "in_kwds": tl in kwds.lower(),
            "in_vocab": any(tl == v.lower() for v in vocab),
            "freq": ta.count(tl) + body.count(tl),
            "is_disease": bool(re.search(_DIS_HEAD, tl, re.I)),
            "is_proc": bool(_PROC_IMG.search(t)),
            "is_anat": bool(_ANATOMY.search(t)),
            "source": source,
            "title_pos": (title.lower().find(tl) / max(1, len(title))) if in_title else 1.0,
        })

    # 1) disease phrases in the TITLE  (strongest signal)
    for m in _DISEASE_PHRASE.finditer(title):
        add(m.group(1), "title_disease", True)
    # 2) procedures / imaging / anatomy in the TITLE
    for rgx, src in ((_PROC_IMG, "title_proc"), (_ANATOMY, "title_anat")):
        for m in rgx.finditer(title):
            add(m.group(0), src, True)
    # 3) training-vocabulary terms present in title/abstract/keywords
    for v in vocab:
        vl = v.lower()
        if len(vl) > 4 and (vl in ta or vl in kwds.lower()):
            add(v, "vocab", vl in title.lower())
    # 4) disease phrases in the ABSTRACT
    for m in _DISEASE_PHRASE.finditer(abstract[:1200]):
        add(m.group(1), "abs_disease", False)
    # 5) procedures / anatomy in the abstract
    for rgx, src in ((_PROC_IMG, "abs_proc"), (_ANATOMY, "abs_anat")):
        for m in rgx.finditer(abstract[:1200]):
            add(m.group(0), src, False)
    # 6) the paper's own keywords
    for kw in re.split(r"[;,]", kwds):
        kw = kw.strip()
        if 3 < len(kw) < 45:
            add(kw, "keyword", kw.lower() in title.lower())
    return cands


COND_N_FEATURES = 13


def condition_features(c, proto_sims):
    return [
        proto_sims.get("centroid", 0.0),
        proto_sims.get("best_cluster", 0.0),
        1.0 if c["in_title"] else 0.0,
        1.0 if c["in_abstract"] else 0.0,
        1.0 if c["in_kwds"] else 0.0,
        1.0 if c["in_vocab"] else 0.0,
        min(c["freq"], 12) / 12.0,
        1.0 if c["is_disease"] else 0.0,
        1.0 if c["is_proc"] else 0.0,
        1.0 if c["is_anat"] else 0.0,
        min(len(c["term"].split()), 5) / 5.0,
        1.0 - min(c["title_pos"], 1.0),          # earlier in title = more central
        min(len(c["term"]), 40) / 40.0,
    ]


def _encode(texts, bs=64):
    if not texts or not _BERT.ok:
        return None
    try:
        return _BERT.model.encode(texts, convert_to_numpy=True,
                                  normalize_embeddings=True,
                                  show_progress_bar=False, batch_size=bs)
    except Exception:
        return None


def build_condition_prototypes(train_part, n_clusters=8):
    """What do real gold condition strings look like (in embedding space)?"""
    golds = [clean_text(g) for g in train_part["conditions"] if clean_text(g)]
    golds = [g for g in golds if len(g) > 2]
    embs = _encode(golds[:400], bs=64)
    if embs is None:
        return None
    centroid = embs.mean(axis=0)
    centroid /= (np.linalg.norm(centroid) + 1e-9)
    clusters = None
    try:
        from sklearn.cluster import KMeans
        k = min(n_clusters, max(2, len(embs) // 20))
        km = KMeans(n_clusters=k, n_init=4, random_state=42).fit(embs)
        clusters = km.cluster_centers_
        clusters /= (np.linalg.norm(clusters, axis=1, keepdims=True) + 1e-9)
    except Exception:
        pass
    return {"centroid": centroid, "clusters": clusters}


def train_condition_ranker(train_part, docs, vocab, protos, n_papers=380, verbose=True):
    """Target = the TRUE metric: BioBERT cosine( repr([term]) , gold_string )."""
    if not _BERT.ok:
        print("[cond-ranker] BioBERT unavailable -> disabled.")
        return None
    sub = train_part.head(n_papers)
    all_feats, all_targets = [], []
    if verbose:
        print(f"[cond-ranker] building training signal from {len(sub)} papers ...")
    for n, (_, r) in enumerate(sub.iterrows()):
        pid = r["pmcid_clean"]
        doc = docs.get(pid)
        if doc is None:
            continue
        gold_str = clean_text(r["conditions"])
        if not gold_str:
            continue
        cands = generate_condition_candidates(doc, vocab)
        if not cands:
            continue
        cand_reprs = [repr([c["term"]]) for c in cands]
        # batch-encode: candidates + the gold, then cosine = dot (already normed)
        embs = _encode(cand_reprs + [gold_str], bs=64)
        if embs is None:
            continue
        cand_embs, gold_emb = embs[:-1], embs[-1]
        targets = cand_embs @ gold_emb                 # the real metric
        for i, c in enumerate(cands):
            ps = {"centroid": float(np.dot(cand_embs[i], protos["centroid"]))
                  if protos else 0.0,
                  "best_cluster": float(np.max(protos["clusters"] @ cand_embs[i]))
                  if (protos and protos["clusters"] is not None) else 0.0}
            all_feats.append(condition_features(c, ps))
            all_targets.append(float(targets[i]))
        if verbose and (n + 1) % 80 == 0:
            print(f"  ... {n + 1}/{len(sub)} papers, {len(all_feats)} labelled terms")
    if len(all_feats) < 200:
        print(f"[cond-ranker] too few samples ({len(all_feats)}) -> disabled.")
        return None
    X = np.asarray(all_feats, float)
    y = np.asarray(all_targets, float)
    try:
        from sklearn.ensemble import GradientBoostingRegressor
        m = GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                      learning_rate=0.06, subsample=0.9,
                                      random_state=42).fit(X, y)
        if verbose:
            print(f"[cond-ranker] trained on {len(X)} terms. train R^2={m.score(X, y):.3f}")
        return m
    except Exception as e:
        print(f"[cond-ranker] failed: {e!r}")
        return None


def optimise_fallback(train_part, docs, vocab, verbose=True):
    """['Disease'] is a guess. Find the fallback string that actually scores best
    on the training papers where we can generate no candidates."""
    if not _BERT.ok:
        return "['Disease']"
    options = ["['Disease']", "['Cancer']", "['Neoplasms']", "['Healthy']",
               "['Disease', 'Cancer']", "['Carcinoma']", "['Stroke']",
               "['Cardiovascular Diseases']", "['Infection']"]
    golds = []
    for _, r in train_part.iterrows():
        doc = docs.get(r["pmcid_clean"])
        if doc is None:
            continue
        if not generate_condition_candidates(doc, vocab):   # only the no-candidate papers
            g = clean_text(r["conditions"])
            if g:
                golds.append(g)
    if len(golds) < 5:                                      # too few -> use overall golds
        golds = [clean_text(g) for g in train_part["conditions"] if clean_text(g)][:200]
    gold_embs = _encode(golds, bs=64)
    opt_embs = _encode(options, bs=64)
    if gold_embs is None or opt_embs is None:
        return "['Disease']"
    means = (opt_embs @ gold_embs.T).mean(axis=1)
    best_i = int(np.argmax(means))
    if verbose:
        print(f"[fallback] tested on {len(golds)} papers:")
        for o, m in sorted(zip(options, means), key=lambda x: -x[1])[:4]:
            print(f"    {o:<32} mean sim = {m:.3f}")
        print(f">>> best fallback = {options[best_i]} (was ['Disease'])")
    return options[best_i]


def conditions_learned(doc, vocab, ranker, protos, top_k, fallback):
    cands = generate_condition_candidates(doc, vocab)
    if not cands or ranker is None:
        return fallback
    reprs = [repr([c["term"]]) for c in cands]
    embs = _encode(reprs, bs=64)
    if embs is None:
        return fallback
    feats = []
    for i, c in enumerate(cands):
        ps = {"centroid": float(np.dot(embs[i], protos["centroid"])) if protos else 0.0,
              "best_cluster": float(np.max(protos["clusters"] @ embs[i]))
              if (protos and protos["clusters"] is not None) else 0.0}
        feats.append(condition_features(c, ps))
    try:
        scores = ranker.predict(np.asarray(feats, float))
    except Exception:
        return fallback
    order = np.argsort(scores)[::-1][:top_k]
    terms = list(dict.fromkeys(cands[i]["term"] for i in order))
    if not terms:
        return fallback
    return repr(terms)


# ============================================================================
# SECTION 16 — v11 PREDICTION ASSEMBLY (conditions mode switchable)
# ============================================================================
def predict_frame_v11(df, docs, vocab, study_default, elig_cap,
                      elig_mode="base", ranker=None, protos=None, top_k=6,
                      age_mode="v2",
                      cond_mode="base", cond_ranker=None, cond_protos=None,
                      cond_topk=2, cond_fallback="['Disease']"):
    rows = []
    for _, r in df.iterrows():
        doc = docs[r["pmcid_clean"]]

        elig = None
        if elig_mode == "learned":
            elig = eligibility_learned(doc, ranker, protos, top_k, elig_cap)
        if elig is None:
            elig = extract_eligibility(doc, elig_cap, mode="base")

        if cond_mode == "learned":
            cond = conditions_learned(doc, vocab, cond_ranker, cond_protos,
                                      cond_topk, cond_fallback)
        else:
            cond = extract_conditions(doc, vocab)

        src = elig if elig != "Not Specified" else doc["full_text"][:3000]
        mn, mx = (extract_age_v2(src, doc) if age_mode == "v2"
                  else extract_age(src, doc))

        rows.append({
            "pmcids": clean_pmcid(r["pmcids"]),
            "conditions": cond,
            "study_type": study_type_predict(doc, study_default),
            "sex": extract_sex(doc),
            "minimum_age": mn,
            "maximum_age": mx,
            "eligibility_criteria": elig,
        })
    return pd.DataFrame(rows)


# ============================================================================
# SECTION 17 — CONDITIONS A/B ON HOLDOUT (safety guard)
# ============================================================================
print("\n" + "=" * 74)
print("STEP 5 — CONDITIONS: learned ranker vs base rules (holdout)")
print("=" * 74)

cond_protos = build_condition_prototypes(tr)
print(f"[cond-proto] {'ready' if cond_protos else 'unavailable'}")
COND_FALLBACK = optimise_fallback(tr, train_docs, vocab_tr)
cond_ranker = train_condition_ranker(tr, train_docs, vocab_tr, cond_protos,
                                     n_papers=COND_RANKER_PAPERS)

# v12 replaces v11's conditions A/B and submission below.
# We still need the FULL-training-data objects for the final test prediction:
cond_protos_full = build_condition_prototypes(train_df)
COND_FALLBACK_FULL = optimise_fallback(train_df, train_docs, vocab_full, verbose=False)
cond_ranker_full = train_condition_ranker(train_df, train_docs, vocab_full,
                                          cond_protos_full,
                                          n_papers=COND_RANKER_PAPERS_FINAL)
print(f"[cond] full-data objects ready (fallback={COND_FALLBACK_FULL})")


# ============================================================================
# SECTION 19 — v12 CACHED RANKING  (makes a WIDE sweep affordable)
# ============================================================================
# v11 re-embedded every candidate for every (top_k, cap) combination, which is
# why we could only afford a narrow sweep. Here we rank each paper's candidates
# ONCE, cache the ranked list, and then sweeping (top_k x cap) is just cheap
# string assembly. This lets us test much longer outputs.
#
# WHY LONGER: the v11 sweep rose monotonically with the cap (400 -> 1400) and
# 1400 was simply the largest value we tried. FM3S rewards noun/verb/word
# overlap with the gold, so a longer answer has more chances to match. We now
# test out to 3000 characters and up to 24 passages.
# ============================================================================
def rank_candidates_once(doc, ranker, protos):
    """Rank a paper's candidate passages ONCE. Returns ranked list (best first)."""
    cands = generate_candidates(doc)
    if not cands or ranker is None:
        return None
    embs = embed_candidates(cands)
    feats = [candidate_features(c, proto_sims_for(embs, protos, i))
             for i, c in enumerate(cands)]
    try:
        scores = ranker.predict(np.asarray(feats, dtype=float))
    except Exception:
        return None
    order = np.argsort(scores)[::-1]
    return [cands[i] for i in order]


def assemble_eligibility(ranked, top_k, cap):
    """Cheap: build the answer from an already-ranked candidate list."""
    if not ranked:
        return None
    chosen = ranked[:top_k]
    incl, excl = [], []
    for c in chosen:
        (excl if EXCL_CUE_RE.search(c["text"]) else incl).append(c["text"])
    parts = []
    if incl:
        parts.append("Inclusion Criteria: " + " ".join(incl))
    if excl:
        parts.append("Exclusion Criteria: " + " ".join(excl))
    text = clean_text(" ".join(parts))
    return text[:cap] if text else None


def build_ranked_cache(pmcids, docs, ranker, protos, label=""):
    cache = {}
    for n, pid in enumerate(pmcids):
        cache[pid] = rank_candidates_once(docs[pid], ranker, protos)
        if label and (n + 1) % 150 == 0:
            print(f"  [{label}] ranked {n + 1}/{len(pmcids)} papers")
    return cache


# ============================================================================
# SECTION 20 — v12 PREDICTION (uses the cached ranking)
# ============================================================================
def predict_frame_v12(df, docs, vocab, study_default, elig_cap,
                      ranked_cache=None, top_k=6, age_mode="v2",
                      cond_rescue=False, cond_ranker=None, cond_protos=None,
                      cond_conf=0.50, cond_fallback="['Disease']"):
    rows = []
    for _, r in df.iterrows():
        pid = r["pmcid_clean"]
        doc = docs[pid]

        elig = None
        if ranked_cache is not None:
            elig = assemble_eligibility(ranked_cache.get(pid), top_k, elig_cap)
        if elig is None:
            elig = extract_eligibility(doc, elig_cap, mode="base")

        # CONDITIONS: base rules everywhere. Only if base gives up (['Disease'])
        # do we consult the learned ranker -- and even then only if it is
        # CONFIDENT. v11 proved a wrong specific guess scores WORSE than the
        # generic hedge, so we never overwrite a real base answer.
        cond = extract_conditions(doc, vocab)
        if cond_rescue and cond in ("['Disease']", "Disease") and cond_ranker is not None:
            rescued = conditions_rescue(doc, vocab, cond_ranker, cond_protos,
                                        cond_conf, cond_fallback)
            if rescued:
                cond = rescued

        src = elig if elig != "Not Specified" else doc["full_text"][:3000]
        mn, mx = (extract_age_v2(src, doc) if age_mode == "v2"
                  else extract_age(src, doc))

        rows.append({
            "pmcids": clean_pmcid(r["pmcids"]),
            "conditions": cond,
            "study_type": study_type_predict(doc, study_default),
            "sex": extract_sex(doc),
            "minimum_age": mn,
            "maximum_age": mx,
            "eligibility_criteria": elig,
        })
    return pd.DataFrame(rows)


def conditions_rescue(doc, vocab, cond_ranker, cond_protos, conf_thr, fallback):
    """Only used when the base rules produced ['Disease'].
    Returns a term ONLY if the ranker is confident; otherwise None (keep hedge)."""
    cands = generate_condition_candidates(doc, vocab)
    if not cands:
        return None
    reprs = [repr([c["term"]]) for c in cands]
    embs = _encode(reprs, bs=64)
    if embs is None:
        return None
    feats = []
    for i, c in enumerate(cands):
        ps = {"centroid": float(np.dot(embs[i], cond_protos["centroid"]))
              if cond_protos else 0.0,
              "best_cluster": float(np.max(cond_protos["clusters"] @ embs[i]))
              if (cond_protos and cond_protos["clusters"] is not None) else 0.0}
        feats.append(condition_features(c, ps))
    try:
        scores = cond_ranker.predict(np.asarray(feats, float))
    except Exception:
        return None
    best = int(np.argmax(scores))
    # the ranker predicts the BioBERT cosine this term would score against gold.
    # ['Disease'] is worth ~0.38, so only swap it out if we clearly beat that.
    if float(scores[best]) < conf_thr:
        return None
    return repr([cands[best]["term"]])


# ============================================================================
# SECTION 21 — v12 EXECUTION: wide eligibility sweep + gated conditions rescue
# ============================================================================
print("\n" + "=" * 74)
print("v12 STEP A — cache the candidate ranking (once per paper)")
print("=" * 74)
va_pmcids_v12 = list(va["pmcid_clean"])
ranked_va = build_ranked_cache(va_pmcids_v12, train_docs, ranker, protos, label="holdout")
n_ok = sum(1 for v in ranked_va.values() if v)
print(f"[cache] ranked {n_ok}/{len(va_pmcids_v12)} holdout papers")

print("\n" + "=" * 74)
print("v12 STEP B — WIDE eligibility sweep (longer answers + more passages)")
print("=" * 74)
print("(v11 stopped at cap=1400 and the score was still RISING, so we go further)")

V12_MODE, V12_TOPK, V12_CAP = "base", BEST_TOPK, BEST_CAP_V10
v12_best = -1.0

# reference: the v11 winner, recomputed here so the comparison is apples-to-apples
ref_v12 = predict_frame_v12(va, train_docs, vocab_tr, study_default_tr, BEST_CAP_V10,
                            ranked_cache=ranked_va, top_k=BEST_TOPK, age_mode=AGE_MODE)
ref_tot, ref_br = score_frame(ref_v12, va_gold)
print(f"\nv11 WINNER (top_k={BEST_TOPK}, cap={BEST_CAP_V10}): composite={ref_tot:.4f} "
      f"elig={ref_br['elig']:.3f}")
v12_best, V12_TOPK, V12_CAP = ref_tot, BEST_TOPK, BEST_CAP_V10

print()
for top_k in TOPK_CANDIDATES:
    for cap in ELIG_CAP_WIDE:
        vp = predict_frame_v12(va, train_docs, vocab_tr, study_default_tr, cap,
                               ranked_cache=ranked_va, top_k=top_k, age_mode=AGE_MODE)
        tot, br = score_frame(vp, va_gold)
        flag = ""
        if tot > v12_best:
            v12_best, V12_TOPK, V12_CAP = tot, top_k, cap
            flag = "  <-- new best"
        print(f"top_k={top_k:>2} cap={cap:>4} | composite={tot:.4f} "
              f"elig={br['elig']:.3f} num={br['num']:.3f}{flag}")

print(f"\n>>> BEST ELIGIBILITY: top_k={V12_TOPK}  cap={V12_CAP}  composite={v12_best:.4f}")
if v12_best > ref_tot + 1e-9:
    print(f"    (longer/more passages HELPED: {ref_tot:.4f} -> {v12_best:.4f})")
else:
    print(f"    (no gain from going longer; keeping the v11 setting)")

print("\n" + "=" * 74)
print("v12 STEP C — CONDITIONS: confidence-gated rescue of ['Disease'] rows only")
print("=" * 74)
print("v11 lesson: a WRONG specific guess scores WORSE than the generic hedge")
print("(['Disease'] is worth ~0.38 cosine). So we only replace it when the ranker")
print("is confident, and we NEVER overwrite an answer the base rules already found.\n")

COND_RESCUE, COND_CONF = False, 0.50
best_with_cond = v12_best
if cond_ranker is not None:
    for conf in [0.40, 0.45, 0.50, 0.55, 0.60]:
        vp = predict_frame_v12(va, train_docs, vocab_tr, study_default_tr, V12_CAP,
                               ranked_cache=ranked_va, top_k=V12_TOPK, age_mode=AGE_MODE,
                               cond_rescue=True, cond_ranker=cond_ranker,
                               cond_protos=cond_protos, cond_conf=conf,
                               cond_fallback=COND_FALLBACK)
        tot, br = score_frame(vp, va_gold)
        n_fb = int(vp["conditions"].isin(["['Disease']", "Disease"]).sum())
        print(f"conf>={conf:.2f} | composite={tot:.4f}  cond={br['cond']:.3f}  "
              f"fallbacks left={n_fb}/{len(vp)}")
        if tot > best_with_cond + 1e-9:
            best_with_cond, COND_RESCUE, COND_CONF = tot, True, conf

print(f"\n>>> CONDITIONS RESCUE = {COND_RESCUE}"
      + (f" (conf>={COND_CONF:.2f})" if COND_RESCUE else " (rescue did not help -> hedge kept)"))

print("\n" + "=" * 74)
print("FINAL HOLDOUT (v12)")
print("=" * 74)
vp_v12 = predict_frame_v12(va, train_docs, vocab_tr, study_default_tr, V12_CAP,
                           ranked_cache=ranked_va, top_k=V12_TOPK, age_mode=AGE_MODE,
                           cond_rescue=COND_RESCUE, cond_ranker=cond_ranker,
                           cond_protos=cond_protos, cond_conf=COND_CONF,
                           cond_fallback=COND_FALLBACK)
t12, b12 = score_frame(vp_v12, va_gold)
print(f"composite = {t12:.4f}      (v11 was {fin_total:.4f})")
print(f"  eligibility (50%) = {b12['elig']:.3f}   (v11: {fin_br['elig']:.3f})")
print(f"  conditions  (20%) = {b12['cond']:.3f}   (v11: {fin_br['cond']:.3f})")
print(f"  study_type  (15%) = {b12['study']:.3f}")
print(f"  age/number  (15%) = {b12['num']:.3f}")


# ============================================================================
# SECTION 22 — FINAL: refit on ALL training data, predict test, write submission
# ============================================================================
print("\n" + "=" * 74)
print("v12 STEP D — predict test + write submission.csv")
print("=" * 74)
print("[cache] ranking test papers (this is the slow part) ...")
test_pmcids_v12 = list(test_df["pmcid_clean"])
ranked_test = build_ranked_cache(test_pmcids_v12, test_docs, ranker_full, protos_full,
                                 label="test")
n_ok_t = sum(1 for v in ranked_test.values() if v)
print(f"[cache] ranked {n_ok_t}/{len(test_pmcids_v12)} test papers")

submission = predict_frame_v12(test_df, test_docs, vocab_full, study_default_full,
                               V12_CAP, ranked_cache=ranked_test, top_k=V12_TOPK,
                               age_mode=AGE_MODE, cond_rescue=COND_RESCUE,
                               cond_ranker=cond_ranker_full, cond_protos=cond_protos_full,
                               cond_conf=COND_CONF, cond_fallback=COND_FALLBACK_FULL)

submission = submission[["pmcids", "conditions", "study_type", "sex",
                         "minimum_age", "maximum_age", "eligibility_criteria"]]
for col in submission.columns:
    submission[col] = submission[col].fillna("Not Specified").astype(str).map(clean_text)
    submission.loc[submission[col].str.strip() == "", col] = "Not Specified"
submission.loc[submission["conditions"].str.strip().isin(["", "[]", "Not Specified"]),
               "conditions"] = COND_FALLBACK_FULL

submission.to_csv(SUBMISSION_PATH, index=False)
check = pd.read_csv(SUBMISSION_PATH)

print("\n" + "=" * 74)
print("SUBMISSION WRITTEN:", SUBMISSION_PATH)
print("=" * 74)
print(f"eligibility: learned ranker | top_k={V12_TOPK} | cap={V12_CAP}")
print(f"conditions : base rules + rescue={COND_RESCUE}"
      + (f" (conf>={COND_CONF:.2f})" if COND_RESCUE else ""))
print(f"age        : {AGE_MODE}")
print(f"\nHOLDOUT composite = {t12:.4f}")
print("Shape:", check.shape, "| NaNs:", int(check.isna().sum().sum()))
print("All 6 columns present:",
      sorted(check.columns.tolist()) == sorted(["pmcids", "conditions", "study_type",
                                                "sex", "minimum_age", "maximum_age",
                                                "eligibility_criteria"]))
print("Empty cells:",
      int((check.astype(str).apply(lambda s: s.str.strip()) == "").sum().sum()))
_fb = int(check["conditions"].isin(["['Disease']", "Disease"]).sum())
print(f"conditions fallback rows: {_fb}/500")
_elen = check["eligibility_criteria"].astype(str).str.len()
print(f"eligibility length: mean={_elen.mean():.0f} max={_elen.max()} chars")
print("\ntop conditions:\n", check["conditions"].value_counts().head(10))
print("\nNO external data. NO API calls. All 6 columns filled. Rule-compliant.")

[warn] scispaCy 'en_ner_bc5cdr_md' not available -> conditions use title/abstract matching (still works offline). (OSError)
[warn] word2number missing (spelled-out numbers skipped; minor).


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[ok] BioBERT loaded: pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb
Train: (416, 7) | Test: (500, 7)
XML mapped: 950
Parsed: 416 train / 500 test

[study_type training prior]
 study_type
INTERVENTIONAL    206
OBSERVATIONAL     118
Name: count, dtype: int64

ELIGIBILITY: A/B test extraction MODE x cap sweep (holdout)

-- mode = base --
cap= 400 | composite=0.6780 | elig=0.707 cond=0.555 study=0.872 num=0.551
cap= 600 | composite=0.6839 | elig=0.722 cond=0.555 study=0.872 num=0.540
cap= 800 | composite=0.6856 | elig=0.731 cond=0.555 study=0.872 num=0.524
cap=1000 | composite=0.6887 | elig=0.737 cond=0.555 study=0.872 num=0.524
cap=1400 | composite=0.6900 | elig=0.743 cond=0.555 study=0.872 num=0.513

-- mode = layout --
cap= 400 | composite=0.6800 | elig=0.708 cond=0.555 study=0.872 num=0.560
cap= 600 | composite=0.6864 | elig=0.723 cond=0.555 study=0.872 num=0.554
cap= 800 | composite=0.6877 | elig=0.727 cond=0.555 study=0.872 num=0.549
cap=1000 | composite=0.6896 | elig=0.731 